In [1]:
import polars as pl

# Configure Polars so Jupyter shows ALL columns and a controlled
# number of rows. This prevents truncation and keeps rendering fast.

pl.Config.set_tbl_cols(-1)   # show all columns, no matter how many
pl.Config.set_tbl_rows(150)   # show only 10 rows in the HTML output

# Load the unified tract-quarter feature dataset
df = pl.read_parquet("../../data/processed/feature_engineering/tract_quarter_features.parquet")

In [2]:
# Extract column names between 'x' and 'y' (exclusive)
start_idx = df.columns.index("weighted_dry_bulb_temp_mean")
end_idx = df.columns.index("severe_weather_clusters")
middle_cols = df.columns[start_idx + 1 : end_idx]

print(middle_cols)

['weighted_dry_bulb_temp_median', 'weighted_dry_bulb_temp_std', 'weighted_dry_bulb_temp_min', 'weighted_dry_bulb_temp_max', 'weighted_dry_bulb_temp_range', 'weighted_dry_bulb_temp_cv', 'weighted_dry_bulb_temp_p10', 'weighted_dry_bulb_temp_p25', 'weighted_dry_bulb_temp_p75', 'weighted_dry_bulb_temp_p90', 'weighted_dry_bulb_temp_bucket_-999_0_hours', 'weighted_dry_bulb_temp_bucket_-999_0_prop', 'weighted_dry_bulb_temp_bucket_0_10_hours', 'weighted_dry_bulb_temp_bucket_0_10_prop', 'weighted_dry_bulb_temp_bucket_10_20_hours', 'weighted_dry_bulb_temp_bucket_10_20_prop', 'weighted_dry_bulb_temp_bucket_20_30_hours', 'weighted_dry_bulb_temp_bucket_20_30_prop', 'weighted_dry_bulb_temp_bucket_30_40_hours', 'weighted_dry_bulb_temp_bucket_30_40_prop', 'weighted_dry_bulb_temp_bucket_40_50_hours', 'weighted_dry_bulb_temp_bucket_40_50_prop', 'weighted_dry_bulb_temp_bucket_50_60_hours', 'weighted_dry_bulb_temp_bucket_50_60_prop', 'weighted_dry_bulb_temp_bucket_60_70_hours', 'weighted_dry_bulb_temp_buc

# Dry Bulb Temperature

In [4]:
dry_bulb_summary = [
    "weighted_dry_bulb_temp_median",
    "weighted_dry_bulb_temp_std",
    "weighted_dry_bulb_temp_min",
    "weighted_dry_bulb_temp_max",
    "weighted_dry_bulb_temp_range",
    "weighted_dry_bulb_temp_cv",
]

# Descriptive stats
print(df.select(dry_bulb_summary).describe())

# Correlation with reliability_index
print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in dry_bulb_summary
    ])
)


shape: (9, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ weighted_dry ┆ weighted_dry ┆ weighted_dr ┆ weighted_dr ┆ weighted_dr ┆ weighted_dr │
│ ---        ┆ _bulb_temp_m ┆ _bulb_temp_s ┆ y_bulb_temp ┆ y_bulb_temp ┆ y_bulb_temp ┆ y_bulb_temp │
│ str        ┆ edian        ┆ td           ┆ _min        ┆ _max        ┆ _range      ┆ _cv         │
│            ┆ ---          ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│            ┆ f64          ┆ f64          ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 10589.0      ┆ 10589.0      ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 49.180324    ┆ 12.880746    ┆ 9.112202    ┆ 83.671036   ┆ 74.5

In [5]:
dry_bulb_percentiles = [
    "weighted_dry_bulb_temp_p10",
    "weighted_dry_bulb_temp_p25",
    "weighted_dry_bulb_temp_p75",
    "weighted_dry_bulb_temp_p90",
]

print(df.select(dry_bulb_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in dry_bulb_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_dry_bulb_t ┆ weighted_dry_bulb_t ┆ weighted_dry_bulb_ ┆ weighted_dry_bulb_ │
│ ---        ┆ emp_p10             ┆ emp_p25             ┆ temp_p75           ┆ temp_p90           │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 31.911102           ┆ 40.837768           ┆ 57.332459          ┆ 65.021725          │
│ std        ┆ 18.454948           ┆ 16.798537           ┆ 15.403367         

In [6]:
dry_bulb_bucket_counts = [
    c for c in df.columns
    if "weighted_dry_bulb_temp_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in dry_bulb_bucket_counts
])


weighted_dry_bulb_temp_bucket_-999_0_hours,weighted_dry_bulb_temp_bucket_0_10_hours,weighted_dry_bulb_temp_bucket_10_20_hours,weighted_dry_bulb_temp_bucket_20_30_hours,weighted_dry_bulb_temp_bucket_30_40_hours,weighted_dry_bulb_temp_bucket_40_50_hours,weighted_dry_bulb_temp_bucket_50_60_hours,weighted_dry_bulb_temp_bucket_60_70_hours,weighted_dry_bulb_temp_bucket_70_80_hours,weighted_dry_bulb_temp_bucket_80_999_hours
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.125152,-0.090589,-0.090484,-0.07247,-0.000134,0.010657,0.01414,0.040741,0.050637,0.036172


In [7]:
dry_bulb_bucket_props = [
    c for c in df.columns
    if "weighted_dry_bulb_temp_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in dry_bulb_bucket_props
])


weighted_dry_bulb_temp_bucket_-999_0_prop,weighted_dry_bulb_temp_bucket_0_10_prop,weighted_dry_bulb_temp_bucket_10_20_prop,weighted_dry_bulb_temp_bucket_20_30_prop,weighted_dry_bulb_temp_bucket_30_40_prop,weighted_dry_bulb_temp_bucket_40_50_prop,weighted_dry_bulb_temp_bucket_50_60_prop,weighted_dry_bulb_temp_bucket_60_70_prop,weighted_dry_bulb_temp_bucket_70_80_prop,weighted_dry_bulb_temp_bucket_80_999_prop
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.125495,-0.091354,-0.091886,-0.074098,-0.002325,0.013137,0.01918,0.04662,0.051691,0.040143


# Wet Bulb Temperature

In [8]:
wet_bulb_summary = [
    "weighted_wet_bulb_temp_mean",
    "weighted_wet_bulb_temp_median",
    "weighted_wet_bulb_temp_std",
    "weighted_wet_bulb_temp_min",
    "weighted_wet_bulb_temp_max",
    "weighted_wet_bulb_temp_range",
    "weighted_wet_bulb_temp_cv",
]

print(df.select(wet_bulb_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in wet_bulb_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_w ┆ weighted_w ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ et_bulb_te ┆ et_bulb_te ┆ wet_bulb_ ┆ wet_bulb_ ┆ wet_bulb_ ┆ wet_bulb_ ┆ wet_bulb_ │
│ str        ┆ mp_mean    ┆ mp_median  ┆ temp_std  ┆ temp_min  ┆ temp_max  ┆ temp_rang ┆ temp_cv   │
│            ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ e         ┆ ---       │
│            ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ ---       ┆ f64       │
│            ┆            ┆            ┆           ┆           ┆           ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 

In [9]:
wet_bulb_percentiles = [
    "weighted_wet_bulb_temp_p10",
    "weighted_wet_bulb_temp_p25",
    "weighted_wet_bulb_temp_p75",
    "weighted_wet_bulb_temp_p90",
]

print(df.select(wet_bulb_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in wet_bulb_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_wet_bulb_t ┆ weighted_wet_bulb_t ┆ weighted_wet_bulb_ ┆ weighted_wet_bulb_ │
│ ---        ┆ emp_p10             ┆ emp_p25             ┆ temp_p75           ┆ temp_p90           │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 27.502113           ┆ 35.636105           ┆ 51.352735          ┆ 57.073868          │
│ std        ┆ 18.607501           ┆ 17.60934            ┆ 13.875912         

In [10]:
wet_bulb_bucket_counts = [
    c for c in df.columns
    if "weighted_wet_bulb_temp_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in wet_bulb_bucket_counts
])


weighted_wet_bulb_temp_bucket_-999_0_hours,weighted_wet_bulb_temp_bucket_0_10_hours,weighted_wet_bulb_temp_bucket_10_20_hours,weighted_wet_bulb_temp_bucket_20_30_hours,weighted_wet_bulb_temp_bucket_30_40_hours,weighted_wet_bulb_temp_bucket_40_50_hours,weighted_wet_bulb_temp_bucket_50_60_hours,weighted_wet_bulb_temp_bucket_60_70_hours,weighted_wet_bulb_temp_bucket_70_80_hours,weighted_wet_bulb_temp_bucket_80_999_hours
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.105292,-0.00058,-0.057246,-0.057047,-0.02744,0.016554,0.025889,0.053553,0.050343,-0.02863


In [11]:
wet_bulb_bucket_props = [
    c for c in df.columns
    if "weighted_wet_bulb_temp_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in wet_bulb_bucket_props
])


weighted_wet_bulb_temp_bucket_-999_0_prop,weighted_wet_bulb_temp_bucket_0_10_prop,weighted_wet_bulb_temp_bucket_10_20_prop,weighted_wet_bulb_temp_bucket_20_30_prop,weighted_wet_bulb_temp_bucket_30_40_prop,weighted_wet_bulb_temp_bucket_40_50_prop,weighted_wet_bulb_temp_bucket_50_60_prop,weighted_wet_bulb_temp_bucket_60_70_prop,weighted_wet_bulb_temp_bucket_70_80_prop,weighted_wet_bulb_temp_bucket_80_999_prop
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.105777,-0.00206,-0.05893,-0.058509,-0.029267,0.018247,0.031721,0.054944,0.047469,-0.025899


# Dew Point Temperature

In [12]:
dew_point_summary = [
    "weighted_dew_point_temp_mean",
    "weighted_dew_point_temp_median",
    "weighted_dew_point_temp_std",
    "weighted_dew_point_temp_min",
    "weighted_dew_point_temp_max",
    "weighted_dew_point_temp_range",
    "weighted_dew_point_temp_cv",
]

print(df.select(dew_point_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in dew_point_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_d ┆ weighted_d ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ ew_point_t ┆ ew_point_t ┆ dew_point ┆ dew_point ┆ dew_point ┆ dew_point ┆ dew_point │
│ str        ┆ emp_mean   ┆ emp_median ┆ _temp_std ┆ _temp_min ┆ _temp_max ┆ _temp_ran ┆ _temp_cv  │
│            ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ge        ┆ ---       │
│            ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ ---       ┆ f64       │
│            ┆            ┆            ┆           ┆           ┆           ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 

In [13]:
dew_point_percentiles = [
    "weighted_dew_point_temp_p10",
    "weighted_dew_point_temp_p25",
    "weighted_dew_point_temp_p75",
    "weighted_dew_point_temp_p90",
]

print(df.select(dew_point_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in dew_point_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_dew_point_ ┆ weighted_dew_point_ ┆ weighted_dew_point ┆ weighted_dew_point │
│ ---        ┆ temp_p10            ┆ temp_p25            ┆ _temp_p75          ┆ _temp_p90          │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 22.9351             ┆ 31.750099           ┆ 47.177412          ┆ 53.166367          │
│ std        ┆ 19.677741           ┆ 17.41965            ┆ 14.723254         

In [14]:
dew_point_bucket_counts = [
    c for c in df.columns
    if "weighted_dew_point_temp_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in dew_point_bucket_counts
])


weighted_dew_point_temp_bucket_-999_0_hours,weighted_dew_point_temp_bucket_0_10_hours,weighted_dew_point_temp_bucket_10_20_hours,weighted_dew_point_temp_bucket_20_30_hours,weighted_dew_point_temp_bucket_30_40_hours,weighted_dew_point_temp_bucket_40_50_hours,weighted_dew_point_temp_bucket_50_60_hours,weighted_dew_point_temp_bucket_60_70_hours,weighted_dew_point_temp_bucket_70_80_hours,weighted_dew_point_temp_bucket_80_999_hours
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.083956,-0.066827,-0.09954,-0.054097,0.021305,0.030194,0.033897,0.053104,0.037804,-0.021749


In [15]:
dew_point_bucket_props = [
    c for c in df.columns
    if "weighted_dew_point_temp_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in dew_point_bucket_props
])


weighted_dew_point_temp_bucket_-999_0_prop,weighted_dew_point_temp_bucket_0_10_prop,weighted_dew_point_temp_bucket_10_20_prop,weighted_dew_point_temp_bucket_20_30_prop,weighted_dew_point_temp_bucket_30_40_prop,weighted_dew_point_temp_bucket_40_50_prop,weighted_dew_point_temp_bucket_50_60_prop,weighted_dew_point_temp_bucket_60_70_prop,weighted_dew_point_temp_bucket_70_80_prop,weighted_dew_point_temp_bucket_80_999_prop
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.084421,-0.067635,-0.101253,-0.05624,0.019602,0.032465,0.038832,0.053235,0.039408,-0.021729


# Relative Humidity

In [16]:
rh_summary = [
    "weighted_relative_humidity_mean",
    "weighted_relative_humidity_median",
    "weighted_relative_humidity_std",
    "weighted_relative_humidity_min",
    "weighted_relative_humidity_max",
    "weighted_relative_humidity_range",
    "weighted_relative_humidity_cv",
]

print(df.select(rh_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in rh_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_r ┆ weighted_r ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ elative_hu ┆ elative_hu ┆ relative_ ┆ relative_ ┆ relative_ ┆ relative_ ┆ relative_ │
│ str        ┆ midity_mea ┆ midity_med ┆ humidity_ ┆ humidity_ ┆ humidity_ ┆ humidity_ ┆ humidity_ │
│            ┆ …          ┆ …          ┆ std       ┆ min       ┆ max       ┆ ran…      ┆ cv        │
│            ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 

In [17]:
rh_percentiles = [
    "weighted_relative_humidity_p10",
    "weighted_relative_humidity_p25",
    "weighted_relative_humidity_p75",
    "weighted_relative_humidity_p90",
]

print(df.select(rh_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in rh_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_relative_h ┆ weighted_relative_h ┆ weighted_relative_ ┆ weighted_relative_ │
│ ---        ┆ umidity_p10         ┆ umidity_p25         ┆ humidity_p75       ┆ humidity_p90       │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 43.358895           ┆ 55.240449           ┆ 82.632746          ┆ 90.625081          │
│ std        ┆ 6.371239            ┆ 6.318659            ┆ 7.056056          

In [18]:
rh_bucket_counts = [
    c for c in df.columns
    if "weighted_relative_humidity_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in rh_bucket_counts
])


weighted_relative_humidity_bucket_0_30_hours,weighted_relative_humidity_bucket_30_50_hours,weighted_relative_humidity_bucket_50_70_hours,weighted_relative_humidity_bucket_70_80_hours,weighted_relative_humidity_bucket_80_100_hours
f64,f64,f64,f64,f64
-0.182697,-0.091615,0.099264,0.12503,-0.042706


In [19]:
rh_bucket_props = [
    c for c in df.columns
    if "weighted_relative_humidity_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in rh_bucket_props
])


weighted_relative_humidity_bucket_0_30_prop,weighted_relative_humidity_bucket_30_50_prop,weighted_relative_humidity_bucket_50_70_prop,weighted_relative_humidity_bucket_70_80_prop,weighted_relative_humidity_bucket_80_100_prop
f64,f64,f64,f64,f64
-0.181754,-0.088761,0.134229,0.156259,-0.03443


# Wind Speed

In [20]:
wind_speed_summary = [
    "weighted_wind_speed_mean",
    "weighted_wind_speed_median",
    "weighted_wind_speed_std",
    "weighted_wind_speed_min",
    "weighted_wind_speed_max",
    "weighted_wind_speed_range",
    "weighted_wind_speed_cv",
]

print(df.select(wind_speed_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in wind_speed_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_w ┆ weighted_w ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ ind_speed_ ┆ ind_speed_ ┆ wind_spee ┆ wind_spee ┆ wind_spee ┆ wind_spee ┆ wind_spee │
│ str        ┆ mean       ┆ median     ┆ d_std     ┆ d_min     ┆ d_max     ┆ d_range   ┆ d_cv      │
│            ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ mean       ┆ 7.243666   ┆ 6.872386   ┆ 4.370952  ┆ 0.00013   ┆ 23.544848 ┆ 

In [21]:
wind_speed_percentiles = [
    "weighted_wind_speed_p10",
    "weighted_wind_speed_p25",
    "weighted_wind_speed_p75",
    "weighted_wind_speed_p90",
]

print(df.select(wind_speed_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in wind_speed_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_wind_speed ┆ weighted_wind_speed ┆ weighted_wind_spee ┆ weighted_wind_spee │
│ ---        ┆ _p10                ┆ _p25                ┆ d_p75              ┆ d_p90              │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 1.669634            ┆ 3.955203            ┆ 10.032036          ┆ 13.180414          │
│ std        ┆ 1.062301            ┆ 1.453777            ┆ 2.030911          

In [22]:
wind_speed_bucket_counts = [
    c for c in df.columns
    if "weighted_wind_speed_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in wind_speed_bucket_counts
])


weighted_wind_speed_bucket_0_5_hours,weighted_wind_speed_bucket_5_10_hours,weighted_wind_speed_bucket_10_15_hours,weighted_wind_speed_bucket_15_20_hours,weighted_wind_speed_bucket_20_30_hours,weighted_wind_speed_bucket_30_999_hours
f64,f64,f64,f64,f64,f64
-0.188133,0.044754,0.102054,0.091651,0.008133,-0.062644


In [23]:
wind_speed_bucket_props = [
    c for c in df.columns
    if "weighted_wind_speed_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in wind_speed_bucket_props
])


weighted_wind_speed_bucket_0_5_prop,weighted_wind_speed_bucket_5_10_prop,weighted_wind_speed_bucket_10_15_prop,weighted_wind_speed_bucket_15_20_prop,weighted_wind_speed_bucket_20_30_prop,weighted_wind_speed_bucket_30_999_prop
f64,f64,f64,f64,f64,f64
-0.158377,0.099188,0.119181,0.092789,0.007018,-0.062644


# Wind Gust Speed

In [24]:
wind_gust_summary = [
    "weighted_wind_gust_speed_mean",
    "weighted_wind_gust_speed_median",
    "weighted_wind_gust_speed_std",
    "weighted_wind_gust_speed_min",
    "weighted_wind_gust_speed_max",
    "weighted_wind_gust_speed_range",
    "weighted_wind_gust_speed_cv",
]

print(df.select(wind_gust_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in wind_gust_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_w ┆ weighted_w ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ ind_gust_s ┆ ind_gust_s ┆ wind_gust ┆ wind_gust ┆ wind_gust ┆ wind_gust ┆ wind_gust │
│ str        ┆ peed_mean  ┆ peed_media ┆ _speed_st ┆ _speed_mi ┆ _speed_ma ┆ _speed_ra ┆ _speed_cv │
│            ┆ ---        ┆ …          ┆ d         ┆ n         ┆ x         ┆ nge       ┆ ---       │
│            ┆ f64        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 

In [25]:
wind_gust_percentiles = [
    "weighted_wind_gust_speed_p10",
    "weighted_wind_gust_speed_p25",
    "weighted_wind_gust_speed_p75",
    "weighted_wind_gust_speed_p90",
]

print(df.select(wind_gust_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in wind_gust_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_wind_gust_ ┆ weighted_wind_gust_ ┆ weighted_wind_gust ┆ weighted_wind_gust │
│ ---        ┆ speed_p10           ┆ speed_p25           ┆ _speed_p75         ┆ _speed_p90         │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 4.770771            ┆ 6.850489            ┆ 20.354581          ┆ 25.864478          │
│ std        ┆ 3.586613            ┆ 3.594654            ┆ 4.382958          

In [26]:
wind_gust_bucket_counts = [
    c for c in df.columns
    if "weighted_wind_gust_speed_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in wind_gust_bucket_counts
])


weighted_wind_gust_speed_bucket_0_20_hours,weighted_wind_gust_speed_bucket_20_30_hours,weighted_wind_gust_speed_bucket_30_40_hours,weighted_wind_gust_speed_bucket_40_50_hours,weighted_wind_gust_speed_bucket_50_999_hours
f64,f64,f64,f64,f64
0.173242,-0.020534,0.083234,0.035431,-0.014986


In [27]:
wind_gust_bucket_props = [
    c for c in df.columns
    if "weighted_wind_gust_speed_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in wind_gust_bucket_props
])


weighted_wind_gust_speed_bucket_0_20_prop,weighted_wind_gust_speed_bucket_20_30_prop,weighted_wind_gust_speed_bucket_30_40_prop,weighted_wind_gust_speed_bucket_40_50_prop,weighted_wind_gust_speed_bucket_50_999_prop
f64,f64,f64,f64,f64
0.204005,-0.021699,0.082123,0.03434,-0.019859


# Precipitation

In [28]:
precip_summary = [
    "weighted_precipitation_mean",
    "weighted_precipitation_median",
    "weighted_precipitation_std",
    "weighted_precipitation_min",
    "weighted_precipitation_max",
    "weighted_precipitation_range",
    "weighted_precipitation_cv",
]

print(df.select(precip_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in precip_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_p ┆ weighted_p ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ recipitati ┆ recipitati ┆ precipita ┆ precipita ┆ precipita ┆ precipita ┆ precipita │
│ str        ┆ on_mean    ┆ on_median  ┆ tion_std  ┆ tion_min  ┆ tion_max  ┆ tion_rang ┆ tion_cv   │
│            ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ e         ┆ ---       │
│            ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ ---       ┆ f64       │
│            ┆            ┆            ┆           ┆           ┆           ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10577.0    ┆ 10577.0    ┆ 10575.0   ┆ 10577.0   ┆ 10577.0   ┆ 10577.0   ┆ 10575.0   │
│ null_count ┆ 12.0       ┆ 12.0       ┆ 14.0      ┆ 12.0      ┆ 12.0      ┆ 

In [29]:
precip_percentiles = [
    "weighted_precipitation_p10",
    "weighted_precipitation_p25",
    "weighted_precipitation_p75",
    "weighted_precipitation_p90",
]

print(df.select(precip_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in precip_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_precipitat ┆ weighted_precipitat ┆ weighted_precipita ┆ weighted_precipita │
│ ---        ┆ ion_p10             ┆ ion_p25             ┆ tion_p75           ┆ tion_p90           │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10577.0             ┆ 10577.0             ┆ 10577.0            ┆ 10577.0            │
│ null_count ┆ 12.0                ┆ 12.0                ┆ 12.0               ┆ 12.0               │
│ mean       ┆ 0.000831            ┆ 0.001192            ┆ 0.007829           ┆ 0.017814           │
│ std        ┆ 0.002216            ┆ 0.003271            ┆ 0.020739          

In [30]:
precip_bucket_counts = [
    c for c in df.columns
    if "weighted_precipitation_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in precip_bucket_counts
])


weighted_precipitation_bucket_-0.0001_0.0001_hours,weighted_precipitation_bucket_0.0001_0.01_hours,weighted_precipitation_bucket_0.01_0.1_hours,weighted_precipitation_bucket_0.1_0.25_hours,weighted_precipitation_bucket_0.25_0.5_hours,weighted_precipitation_bucket_0.5_999_hours
f64,f64,f64,f64,f64,f64
0.250804,0.146364,0.092844,-0.051287,-0.010692,0.028455


In [31]:
precip_bucket_props = [
    c for c in df.columns
    if "weighted_precipitation_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in precip_bucket_props
])


weighted_precipitation_bucket_-0.0001_0.0001_prop,weighted_precipitation_bucket_0.0001_0.01_prop,weighted_precipitation_bucket_0.01_0.1_prop,weighted_precipitation_bucket_0.1_0.25_prop,weighted_precipitation_bucket_0.25_0.5_prop,weighted_precipitation_bucket_0.5_999_prop
f64,f64,f64,f64,f64,f64
0.271269,0.162117,0.113171,-0.046001,0.000332,0.038855


# Visibility

In [32]:
visibility_summary = [
    "weighted_visibility_mean",
    "weighted_visibility_median",
    "weighted_visibility_std",
    "weighted_visibility_min",
    "weighted_visibility_max",
    "weighted_visibility_range",
    "weighted_visibility_cv",
]

print(df.select(visibility_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in visibility_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_v ┆ weighted_v ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ isibility_ ┆ isibility_ ┆ visibilit ┆ visibilit ┆ visibilit ┆ visibilit ┆ visibilit │
│ str        ┆ mean       ┆ median     ┆ y_std     ┆ y_min     ┆ y_max     ┆ y_range   ┆ y_cv      │
│            ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ mean       ┆ 8.906942   ┆ 9.765393   ┆ 1.914924  ┆ 0.483781  ┆ 9.985647  ┆ 

In [33]:
visibility_percentiles = [
    "weighted_visibility_p10",
    "weighted_visibility_p25",
    "weighted_visibility_p75",
    "weighted_visibility_p90",
]

print(df.select(visibility_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in visibility_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_visibility ┆ weighted_visibility ┆ weighted_visibilit ┆ weighted_visibilit │
│ ---        ┆ _p10                ┆ _p25                ┆ y_p75              ┆ y_p90              │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 6.176058            ┆ 8.988135            ┆ 9.910318           ┆ 9.931158           │
│ std        ┆ 1.571759            ┆ 1.13464             ┆ 0.462869          

In [34]:
visibility_bucket_counts = [
    c for c in df.columns
    if "weighted_visibility_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in visibility_bucket_counts
])


weighted_visibility_bucket_0_1_hours,weighted_visibility_bucket_1_2_hours,weighted_visibility_bucket_2_5_hours,weighted_visibility_bucket_5_999_hours
f64,f64,f64,f64
-0.056198,-0.028963,-0.109677,0.006993


In [35]:
visibility_bucket_props = [
    c for c in df.columns
    if "weighted_visibility_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in visibility_bucket_props
])


weighted_visibility_bucket_0_1_prop,weighted_visibility_bucket_1_2_prop,weighted_visibility_bucket_2_5_prop,weighted_visibility_bucket_5_999_prop
f64,f64,f64,f64
-0.063936,-0.034723,-0.101005,0.092361


# Station Pressure

In [36]:
station_pressure_summary = [
    "weighted_station_pressure_mean",
    "weighted_station_pressure_median",
    "weighted_station_pressure_std",
    "weighted_station_pressure_min",
    "weighted_station_pressure_max",
    "weighted_station_pressure_range",
    "weighted_station_pressure_cv",
]

print(df.select(station_pressure_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in station_pressure_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_s ┆ weighted_s ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ tation_pre ┆ tation_pre ┆ station_p ┆ station_p ┆ station_p ┆ station_p ┆ station_p │
│ str        ┆ ssure_mean ┆ ssure_medi ┆ ressure_s ┆ ressure_m ┆ ressure_m ┆ ressure_r ┆ ressure_c │
│            ┆ ---        ┆ …          ┆ td        ┆ in        ┆ ax        ┆ ang…      ┆ v         │
│            ┆ f64        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 

In [37]:
station_pressure_percentiles = [
    "weighted_station_pressure_p10",
    "weighted_station_pressure_p25",
    "weighted_station_pressure_p75",
    "weighted_station_pressure_p90",
]

print(df.select(station_pressure_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in station_pressure_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_station_pr ┆ weighted_station_pr ┆ weighted_station_p ┆ weighted_station_p │
│ ---        ┆ essure_p10          ┆ essure_p25          ┆ ressure_p75        ┆ ressure_p90        │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 24.991431           ┆ 26.278537           ┆ 28.952487          ┆ 29.082501          │
│ std        ┆ 6.255368            ┆ 5.262359            ┆ 1.327584          

In [38]:
station_pressure_bucket_counts = [
    c for c in df.columns
    if "weighted_station_pressure_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in station_pressure_bucket_counts
])


weighted_station_pressure_bucket_28.0_28.75_hours,weighted_station_pressure_bucket_28.75_29.25_hours,weighted_station_pressure_bucket_29.25_29.75_hours,weighted_station_pressure_bucket_29.75_31.0_hours
f64,f64,f64,f64
-0.272002,0.106067,0.1775,-0.045858


In [39]:
station_pressure_bucket_props = [
    c for c in df.columns
    if "weighted_station_pressure_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in station_pressure_bucket_props
])


weighted_station_pressure_bucket_28.0_28.75_prop,weighted_station_pressure_bucket_28.75_29.25_prop,weighted_station_pressure_bucket_29.25_29.75_prop,weighted_station_pressure_bucket_29.75_31.0_prop
f64,f64,f64,f64
-0.2694,0.117675,0.178053,-0.046929


# Barometric Pressure

In [40]:
baro_summary = [
    "weighted_barometric_pressure_mean",
    "weighted_barometric_pressure_median",
    "weighted_barometric_pressure_std",
    "weighted_barometric_pressure_min",
    "weighted_barometric_pressure_max",
    "weighted_barometric_pressure_range",
    "weighted_barometric_pressure_cv",
]

print(df.select(baro_summary).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in baro_summary
    ])
)


shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ weighted_b ┆ weighted_b ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---        ┆ arometric_ ┆ arometric_ ┆ barometri ┆ barometri ┆ barometri ┆ barometri ┆ barometri │
│ str        ┆ pressure_m ┆ pressure_m ┆ c_pressur ┆ c_pressur ┆ c_pressur ┆ c_pressur ┆ c_pressur │
│            ┆ …          ┆ …          ┆ e_s…      ┆ e_m…      ┆ e_m…      ┆ e_r…      ┆ e_c…      │
│            ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 10589.0    ┆ 10589.0    ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   ┆ 10589.0   │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 

In [41]:
baro_percentiles = [
    "weighted_barometric_pressure_p10",
    "weighted_barometric_pressure_p25",
    "weighted_barometric_pressure_p75",
    "weighted_barometric_pressure_p90",
]

print(df.select(baro_percentiles).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in baro_percentiles
    ])
)


shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ weighted_barometric ┆ weighted_barometric ┆ weighted_barometri ┆ weighted_barometri │
│ ---        ┆ _pressure_p…        ┆ _pressure_p…        ┆ c_pressure_p…      ┆ c_pressure_p…      │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 10589.0             ┆ 10589.0             ┆ 10589.0            ┆ 10589.0            │
│ null_count ┆ 0.0                 ┆ 0.0                 ┆ 0.0                ┆ 0.0                │
│ mean       ┆ 27.673525           ┆ 28.706326           ┆ 29.899862          ┆ 30.02457           │
│ std        ┆ 3.945516            ┆ 2.881692            ┆ 1.328676          

In [42]:
baro_bucket_counts = [
    c for c in df.columns
    if "weighted_barometric_pressure_bucket" in c and c.endswith("_hours")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in baro_bucket_counts
])


weighted_barometric_pressure_bucket_28.0_28.75_hours,weighted_barometric_pressure_bucket_28.75_29.25_hours,weighted_barometric_pressure_bucket_29.25_29.75_hours,weighted_barometric_pressure_bucket_29.75_31.0_hours
f64,f64,f64,f64
0.106809,0.032532,-0.070093,0.05766


In [43]:
baro_bucket_props = [
    c for c in df.columns
    if "weighted_barometric_pressure_bucket" in c and c.endswith("_prop")
]

df.select([
    pl.corr("reliability_index", c).alias(c)
    for c in baro_bucket_props
])


weighted_barometric_pressure_bucket_28.0_28.75_prop,weighted_barometric_pressure_bucket_28.75_29.25_prop,weighted_barometric_pressure_bucket_29.25_29.75_prop,weighted_barometric_pressure_bucket_29.75_31.0_prop
f64,f64,f64,f64
0.10683,0.031338,-0.071887,0.071303


# Engineered Conditional Features

In [ ]:
engineered_features_1 = [
    "temp_below_freezing_hours",
    "temp_above_90_hours",
    "gust_gt_20_hours",
    "gust_gt_30_hours",
    "gust_gt_40_hours",
    "gust_gt_50_hours",
]

engineered_features_2 = [
    "precip_gt_0_01_hours",
    "precip_gt_0_10_hours",
    "precip_gt_0_25_hours",
    "precip_gt_0_50_hours",
    "visibility_lt_1_hours",
    "low_pressure_hours",
]

engineered_features_3 = [
    "wet_windy_hours",
    "cold_windy_hours",
    "hot_humid_hours",
    "storm_event_hours",
    "observed_hours",
]

engineered_features_4 = [
    "expected_hours",
    "coverage_ratio",
    "wind_dir_resultant_length",
    "wind_dir_circular_variance",
    "sustained_high_wind_periods",
]


In [49]:
print(df.select(engineered_features_1).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in engineered_features_1
    ])
)

shape: (9, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ temp_below_f ┆ temp_above_9 ┆ gust_gt_20_ ┆ gust_gt_30_ ┆ gust_gt_40_ ┆ gust_gt_50_ │
│ ---        ┆ reezing_hour ┆ 0_hours      ┆ hours       ┆ hours       ┆ hours       ┆ hours       │
│ str        ┆ s            ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│            ┆ ---          ┆ f64          ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
│            ┆ f64          ┆              ┆             ┆             ┆             ┆             │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 10589.0      ┆ 10589.0      ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 477.845594   ┆ 3.368779     ┆ 223.196336  ┆ 43.283124   ┆ 2.46

In [50]:
print(df.select(engineered_features_2).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in engineered_features_2
    ])
)

shape: (9, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ precip_gt_0_ ┆ precip_gt_0_ ┆ precip_gt_0 ┆ precip_gt_0 ┆ visibility_ ┆ low_pressur │
│ ---        ┆ 01_hours     ┆ 10_hours     ┆ _25_hours   ┆ _50_hours   ┆ lt_1_hours  ┆ e_hours     │
│ str        ┆ ---          ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│            ┆ f64          ┆ f64          ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 10589.0      ┆ 10589.0      ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 90.055529    ┆ 11.90169     ┆ 2.239305    ┆ 0.463878    ┆ 18.430824   ┆ 957.933516  │
│ std        ┆ 41.82181     ┆ 9.823675     ┆ 2.913395    ┆ 1.040792    ┆ 20.4

In [51]:
print(df.select(engineered_features_3).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in engineered_features_3
    ])
)

shape: (9, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ statistic  ┆ low_pressure ┆ wet_windy_ho ┆ cold_windy_ ┆ hot_humid_h ┆ storm_event ┆ observed_ho │
│ ---        ┆ _hours       ┆ urs          ┆ hours       ┆ ours        ┆ _hours      ┆ urs         │
│ str        ┆ ---          ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│            ┆ f64          ┆ f64          ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ count      ┆ 10589.0      ┆ 10589.0      ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     ┆ 10589.0     │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 957.933516   ┆ 18.71206     ┆ 33.610634   ┆ 0.007838    ┆ 0.005383    ┆ 2063.073189 │
│ std        ┆ 568.111168   ┆ 15.909919    ┆ 46.178465   ┆ 0.089255    ┆ 0.08

In [52]:
print(df.select(engineered_features_4).describe())

print(
    df.select([
        pl.corr("reliability_index", c).alias(c)
        for c in engineered_features_4
    ])
)


shape: (9, 6)
┌────────────┬────────────────┬────────────────┬─────────────────┬────────────────┬────────────────┐
│ statistic  ┆ expected_hours ┆ coverage_ratio ┆ wind_dir_result ┆ wind_dir_circu ┆ sustained_high │
│ ---        ┆ ---            ┆ ---            ┆ ant_length      ┆ lar_variance   ┆ _wind_periods  │
│ str        ┆ f64            ┆ f64            ┆ ---             ┆ ---            ┆ ---            │
│            ┆                ┆                ┆ f64             ┆ f64            ┆ f64            │
╞════════════╪════════════════╪════════════════╪═════════════════╪════════════════╪════════════════╡
│ count      ┆ 10589.0        ┆ 10589.0        ┆ 10589.0         ┆ 10589.0        ┆ 10589.0        │
│ null_count ┆ 0.0            ┆ 0.0            ┆ 0.0             ┆ 0.0            ┆ 0.0            │
│ mean       ┆ 2160.0         ┆ 0.955126       ┆ 0.260014        ┆ 0.739986       ┆ 0.252148       │
│ std        ┆ 0.0            ┆ 0.138365       ┆ 0.100235        ┆ 0.100235  